# Manuscript benchmark: 2D triangle L1 (with L2 pre) on every section of the real DVF

This notebook applies the two-stage `L2-multipass -> L1-polish` correction recipe (established in notebooks 17 and 18) to **every z-slice** of the real registration field at `data/corrected_correspondences_count_touching/registered_output/deformation3d.npy`.

**Pipeline per slice.** For each `z in [0, 528)`:

1. Extract the slice as a `(3, 1, H, W)` deformation; channel 0 (dz) is identically zero in this dataset, so each slice is a genuine 2D problem.
2. **L2 phase.** Run `iterative_serial(..., enforce_triangles=True)` -- the package's windowed SLSQP solver, configured to use the 2-triangle constraint (the manuscript's check) rather than central-difference Jdet. Drives `n_neg_tri -> 0`.
3. **L1 polish phase.** Identify connected components of cells *touched* by the L2 phase. For each component, re-solve the bounding sub-window with the smoothed-L1 objective + 2-triangle constraint + frozen-edge boundary. Accept only if no new folds appear and the L1 norm strictly drops.
4. Record one CSV row per slice with init/L2/L1 metrics and timing. Save incrementally so killing the kernel preserves work; on restart the notebook resumes from where it left off.

**Output.** `notebooks/manuscript/output/2d_real_full/`:

- `per_slice.csv` -- the per-slice results table (appended row-by-row).
- `checkpoint.npz` -- snapshot of the corrected `(3, D, H, W)` volume, updated every N slices.
- `summary_*.{pdf,png}` -- aggregate plots once every slice is done.

Companion notebook: `04_benchmark_3d_real_full.ipynb` runs the same recipe with the 6-tet constraint on the full 3D volume.

## Setup

In [ ]:
import os, sys, time, json, traceback
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, NonlinearConstraint
from scipy.ndimage import label as cc_label

from dvfopt import DEFAULT_PARAMS, iterative_serial
from dvfopt.jacobian import triangle_sign_areas2D
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d

# Tunables.
THRESHOLD = DEFAULT_PARAMS['threshold']  # 0.01
ERR_TOL = DEFAULT_PARAMS['err_tol']      # 1e-5
EPS_L1 = 1e-4

# Aggressive budget per user choice.
L2_MAX_OUTER = 20000      # iterative_serial outer-loop cap
L2_MAX_PER_INDEX = 200    # per-pixel inner-loop cap
L2_MAX_MINIMIZE = 1000    # per-window SLSQP iter cap
L1_POLISH_MAX_ITER = 400  # per-component L1 SLSQP cap
L1_TOUCH_TOL = 1e-3       # what counts as 'touched' for L1 polish region selection
L1_BORDER = 1             # 1-px positive border around touched component (frozen edges)
L1_COMPONENT_MAX_SIZE = 6000   # if a component's window exceeds this many cells, skip polish on it

# Checkpoint cadence.
CHECKPOINT_EVERY = 20

# Smoke-test toggle: when True, only run a handful of slices for sanity-check.
SMOKE_TEST = False
SMOKE_SLICE_INDICES = None  # set to a list of ints to override

OUTPUT_DIR = os.path.abspath('output/2d_real_full')
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, 'per_slice.csv')
CKPT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint.npz')
LOG_PATH = os.path.join(OUTPUT_DIR, 'run.log')

DATA_PATH = os.path.abspath(
    os.path.join('..', '..', 'data',
                 'corrected_correspondences_count_touching',
                 'registered_output', 'deformation3d.npy'))

print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')
print(f'output dir: {OUTPUT_DIR}')
print(f'data path : {DATA_PATH}')

## Load the real DVF and survey folds per slice

In [ ]:
phi_full = np.load(DATA_PATH)   # (3, D, H, W) with [dz, dy, dx]
D, H, W = phi_full.shape[1:]
print(f'deformation3d.npy shape : {phi_full.shape}  (D={D}, H={H}, W={W})')
print(f'channel ranges:')
for c, name in enumerate(['dz', 'dy', 'dx']):
    print(f'  {name}: min={phi_full[c].min():+.3f}, max={phi_full[c].max():+.3f}, mean={phi_full[c].mean():+.3f}')
if not np.all(phi_full[0] == 0):
    print('NOTE: channel 0 (dz) is NOT identically zero -- per-slice 2D treatment is an approximation.')
else:
    print('Channel 0 (dz) is identically zero -- per-slice 2D treatment is exact.')

In [ ]:
def slice_fold_stats(phi_full, z):
    """Return (n_neg_tri, min_tri, n_neg_jdet) for slice z."""
    dy = phi_full[1, z]; dx = phi_full[2, z]
    T1, T2 = _triangle_areas_2d(dy, dx)
    n_neg = int((T1 <= 0).sum() + (T2 <= 0).sum())
    return n_neg, float(min(T1.min(), T2.min()))


t0 = time.time()
fold_per_slice = np.zeros(D, dtype=np.int64)
min_tri_per_slice = np.zeros(D, dtype=np.float64)
for z in range(D):
    n_neg, mt = slice_fold_stats(phi_full, z)
    fold_per_slice[z] = n_neg
    min_tri_per_slice[z] = mt
print(f'Surveyed {D} slices in {time.time()-t0:.1f}s')
print(f'slices with folds : {int((fold_per_slice > 0).sum())} / {D}')
print(f'total folded triangles across all slices: {int(fold_per_slice.sum())}')
print(f'max folds in a single slice : {int(fold_per_slice.max())}  (slice z={int(np.argmax(fold_per_slice))})')
print(f'global min triangle area    : {float(min_tri_per_slice.min()):+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), constrained_layout=True)
axes[0].plot(fold_per_slice, lw=0.8)
axes[0].set_xlabel('z (slice index)'); axes[0].set_ylabel('n_neg_tri (initial)')
axes[0].set_title('Initial folded-triangle count per slice')
axes[1].hist(fold_per_slice[fold_per_slice > 0], bins=60, color='#5b7fb5')
axes[1].set_xlabel('n_neg_tri'); axes[1].set_ylabel('# slices')
axes[1].set_title(f'Distribution over the {int((fold_per_slice>0).sum())} folded slices')
plt.show()

## Helpers

Three building blocks:

1. `metrics_2d` -- common metric block (n_neg_tri, min_tri, L1, L2) given two `(2,H,W)` fields.
2. `run_l2_phase` -- thin wrapper around `iterative_serial(enforce_triangles=True)` that also captures wall time.
3. `run_l1_polish_phase` -- the windowed L1 polish: connected components of touched cells, then per-component SLSQP with the smoothed-L1 objective.

In [ ]:
def metrics_2d(phi_now, phi_anchor):
    """phi_now, phi_anchor: shape (2, H, W). Returns dict."""
    T1, T2 = _triangle_areas_2d(phi_now[0], phi_now[1])
    return {
        'n_neg_tri': int((T1 <= 0).sum() + (T2 <= 0).sum()),
        'min_tri': float(min(T1.min(), T2.min())),
        'L1': float(np.abs(phi_now - phi_anchor).sum()),
        'L2': float(np.linalg.norm(phi_now - phi_anchor)),
    }

In [ ]:
def run_l2_phase(deformation_3channel):
    """Run windowed L2 SLSQP with the 2-triangle constraint.

    Parameters
    ----------
    deformation_3channel : ndarray, shape (3, 1, H, W)

    Returns
    -------
    phi_l2 : ndarray, shape (2, H, W)
    info : dict (wall_time, exception or None)
    """
    t0 = time.time()
    info = {'exception': None}
    try:
        phi_l2 = iterative_serial(
            deformation_3channel.copy(),
            enforce_triangles=True,
            threshold=THRESHOLD,
            err_tol=ERR_TOL,
            max_iterations=L2_MAX_OUTER,
            max_per_index_iter=L2_MAX_PER_INDEX,
            max_minimize_iter=L2_MAX_MINIMIZE,
            verbose=0,
        )
    except Exception as exc:
        info['exception'] = f'{type(exc).__name__}: {exc}'
        # Fall back to the input field unchanged so downstream code sees something usable.
        phi_l2 = np.stack([deformation_3channel[1, 0].copy(),
                            deformation_3channel[2, 0].copy()])
    info['wall_time'] = time.time() - t0
    return phi_l2, info

In [ ]:
def _l1_polish_window(phi_window_l2, phi_window_anchor, interior_mask,
                       threshold=THRESHOLD, eps=EPS_L1,
                       max_iter=L1_POLISH_MAX_ITER):
    """Solve smoothed-L1 SLSQP on a single sub-window with frozen edges.

    Variables = phi values at cells where ``interior_mask`` is True
    (these are the cells we are willing to move); cells where the mask is
    False keep their L2 value ("frozen edges"). Constraint = both
    triangle signed areas >= threshold at every cell whose 4 corners lie
    inside the window. Objective = sum-of-smoothed-L1 to the anchor
    (original) field.
    """
    _, h, w = phi_window_l2.shape
    int_idx = np.argwhere(interior_mask)
    n_int = len(int_idx)
    if n_int == 0:
        return phi_window_l2.copy(), {'nit': 0, 'success': True, 'status': 0}
    int_y = int_idx[:, 0]
    int_x = int_idx[:, 1]

    def pack(phi):
        return np.concatenate([phi[0][interior_mask], phi[1][interior_mask]])

    def unpack(z):
        phi = phi_window_l2.copy()
        phi[0, int_y, int_x] = z[:n_int]
        phi[1, int_y, int_x] = z[n_int:]
        return phi

    z_init = pack(phi_window_l2)
    z_anchor = pack(phi_window_anchor)

    def obj(z):
        d = z - z_anchor
        s = np.sqrt(d * d + eps * eps)
        return float(s.sum()), d / s

    def constr(z):
        phi = unpack(z)
        T1, T2 = _triangle_areas_2d(phi[0], phi[1])
        return np.concatenate([T1.flatten(), T2.flatten()])

    nl = NonlinearConstraint(constr, lb=threshold, ub=np.inf)
    res = minimize(obj, z_init, jac=True, method='SLSQP',
                    constraints=[nl],
                    options={'maxiter': max_iter, 'ftol': 1e-9, 'disp': False})
    return unpack(res.x), {'nit': int(res.nit), 'success': bool(res.success),
                            'status': int(res.status)}


def run_l1_polish_phase(phi_l2, phi_anchor):
    """Windowed L1 polish over connected components of touched cells.

    Returns (phi_l1, info) where info contains per-component stats.
    """
    t0 = time.time()
    H_, W_ = phi_l2.shape[1:]
    diff_mag = np.abs(phi_l2 - phi_anchor).max(axis=0)
    touched = diff_mag > L1_TOUCH_TOL

    info = {
        'n_components': 0,
        'n_components_polished': 0,
        'n_components_rejected': 0,
        'n_components_skipped_too_large': 0,
        'iter_total': 0,
        'wall_time': 0.0,
    }
    if not touched.any():
        info['wall_time'] = time.time() - t0
        return phi_l2.copy(), info

    labels, n_comp = cc_label(touched)
    info['n_components'] = int(n_comp)
    phi_l1 = phi_l2.copy()

    for cid in range(1, n_comp + 1):
        cells = np.argwhere(labels == cid)
        y0, x0 = cells.min(axis=0)
        y1, x1 = cells.max(axis=0) + 1
        # Pad by L1_BORDER on each side (frozen edges).
        y0 = max(0, y0 - L1_BORDER); x0 = max(0, x0 - L1_BORDER)
        y1 = min(H_, y1 + L1_BORDER); x1 = min(W_, x1 + L1_BORDER)
        win_h = y1 - y0; win_w = x1 - x0
        if win_h * win_w > L1_COMPONENT_MAX_SIZE:
            info['n_components_skipped_too_large'] += 1
            continue
        # Interior mask: same shape as the window. Edge cells (the outer
        # 1-cell ring) are frozen. Cells outside the touched-component
        # bounding box but inside the window are also frozen.
        interior_mask = np.zeros((win_h, win_w), dtype=bool)
        interior_mask[1:-1, 1:-1] = True
        # Snapshot for global feasibility check.
        phi_l1_before = phi_l1.copy()
        phi_win_l2 = phi_l1[:, y0:y1, x0:x1].copy()
        phi_win_anchor = phi_anchor[:, y0:y1, x0:x1]
        try:
            phi_win_new, win_info = _l1_polish_window(
                phi_win_l2, phi_win_anchor, interior_mask)
        except Exception:
            info['n_components_rejected'] += 1
            continue
        # Test whether the candidate update keeps the slice feasible AND
        # strictly lowers L1 on the window.
        phi_l1_candidate = phi_l1.copy()
        phi_l1_candidate[:, y0:y1, x0:x1] = phi_win_new
        T1c, T2c = _triangle_areas_2d(phi_l1_candidate[0], phi_l1_candidate[1])
        if int((T1c <= 0).sum() + (T2c <= 0).sum()) > 0:
            info['n_components_rejected'] += 1
            continue
        l1_before = float(np.abs(phi_l1_before[:, y0:y1, x0:x1] - phi_win_anchor).sum())
        l1_after = float(np.abs(phi_win_new - phi_win_anchor).sum())
        if l1_after >= l1_before - 1e-9:
            # Polish did not help -- skip.
            info['n_components_rejected'] += 1
            continue
        phi_l1 = phi_l1_candidate
        info['n_components_polished'] += 1
        info['iter_total'] += win_info['nit']

    info['wall_time'] = time.time() - t0
    return phi_l1, info

## Per-slice runner

In [ ]:
def run_one_slice(phi_full, z):
    """Run the full L2 + L1 pipeline on slice z. Returns a result-row dict."""
    deformation = phi_full[:, z:z+1, :, :].copy()  # (3, 1, H, W)
    phi_anchor = np.stack([deformation[1, 0].copy(), deformation[2, 0].copy()])
    init = metrics_2d(phi_anchor, phi_anchor)
    t_total = time.time()
    row = {
        'z': int(z), 'H': int(H), 'W': int(W),
        'init_n_neg_tri': init['n_neg_tri'],
        'init_min_tri': init['min_tri'],
    }
    # Slice with no folds: nothing to do.
    if init['n_neg_tri'] == 0:
        row.update({
            'L2_n_neg_tri': 0, 'L2_min_tri': init['min_tri'],
            'L2_L1': 0.0, 'L2_L2': 0.0, 'L2_t': 0.0, 'L2_exception': None,
            'L1_L1': 0.0, 'L1_L2': 0.0,
            'L1_n_neg_tri': 0, 'L1_min_tri': init['min_tri'],
            'L1_components': 0, 'L1_polished': 0,
            'L1_rejected': 0, 'L1_skipped_large': 0,
            'L1_iter_total': 0, 'L1_t': 0.0,
            'total_t': time.time() - t_total,
            'feasible': True,
            'l1_drop_pct': 0.0,
            'phi_l1': np.stack([deformation[1, 0], deformation[2, 0]]),
        })
        return row
    # L2 phase.
    phi_l2, l2_info = run_l2_phase(deformation)
    l2_m = metrics_2d(phi_l2, phi_anchor)
    row.update({
        'L2_n_neg_tri': l2_m['n_neg_tri'], 'L2_min_tri': l2_m['min_tri'],
        'L2_L1': l2_m['L1'], 'L2_L2': l2_m['L2'],
        'L2_t': l2_info['wall_time'], 'L2_exception': l2_info['exception'],
    })
    # L1 polish phase (only meaningful if L2 reached feasibility).
    if l2_m['n_neg_tri'] == 0:
        phi_l1, l1_info = run_l1_polish_phase(phi_l2, phi_anchor)
        l1_m = metrics_2d(phi_l1, phi_anchor)
    else:
        phi_l1 = phi_l2
        l1_m = l2_m
        l1_info = {'n_components': 0, 'n_components_polished': 0,
                    'n_components_rejected': 0, 'n_components_skipped_too_large': 0,
                    'iter_total': 0, 'wall_time': 0.0}
    row.update({
        'L1_L1': l1_m['L1'], 'L1_L2': l1_m['L2'],
        'L1_n_neg_tri': l1_m['n_neg_tri'], 'L1_min_tri': l1_m['min_tri'],
        'L1_components': l1_info['n_components'],
        'L1_polished': l1_info['n_components_polished'],
        'L1_rejected': l1_info['n_components_rejected'],
        'L1_skipped_large': l1_info['n_components_skipped_too_large'],
        'L1_iter_total': l1_info['iter_total'],
        'L1_t': l1_info['wall_time'],
        'total_t': time.time() - t_total,
        'feasible': bool(l1_m['n_neg_tri'] == 0),
        'l1_drop_pct': (100.0 * (l2_m['L1'] - l1_m['L1']) / l2_m['L1']
                         if l2_m['L1'] > 0 else 0.0),
        'phi_l1': phi_l1,
    })
    return row

## Resumable orchestration

Reads `per_slice.csv` if it exists and skips any `z` already present. Appends one row per slice as it finishes. Snapshots the corrected volume to `checkpoint.npz` every `CHECKPOINT_EVERY` slices.

Each slice is wrapped in `try/except` so a single bad slice cannot crash the whole run -- failures are logged with `feasible=False` and the original field is kept for that slice.

In [ ]:
CSV_COLUMNS = [
    'z', 'H', 'W',
    'init_n_neg_tri', 'init_min_tri',
    'L2_n_neg_tri', 'L2_min_tri', 'L2_L1', 'L2_L2', 'L2_t', 'L2_exception',
    'L1_L1', 'L1_L2', 'L1_n_neg_tri', 'L1_min_tri',
    'L1_components', 'L1_polished', 'L1_rejected', 'L1_skipped_large',
    'L1_iter_total', 'L1_t',
    'total_t', 'feasible', 'l1_drop_pct',
]


def load_done_slices():
    if not os.path.exists(CSV_PATH):
        return set()
    try:
        df = pd.read_csv(CSV_PATH)
        return set(int(z) for z in df['z'].values)
    except Exception:
        return set()


def init_csv_if_needed():
    if not os.path.exists(CSV_PATH):
        with open(CSV_PATH, 'w', newline='') as f:
            f.write(','.join(CSV_COLUMNS) + '\n')


def append_csv_row(row):
    parts = []
    for c in CSV_COLUMNS:
        v = row.get(c, '')
        if v is None:
            parts.append('')
        elif isinstance(v, float):
            parts.append(f'{v:.6g}')
        elif isinstance(v, bool):
            parts.append('True' if v else 'False')
        else:
            s = str(v).replace(',', ';').replace('\n', ' ')
            parts.append(s)
    with open(CSV_PATH, 'a', newline='') as f:
        f.write(','.join(parts) + '\n')


def save_checkpoint(corrected_full):
    tmp = CKPT_PATH + '.tmp'
    np.savez_compressed(tmp, phi_corrected=corrected_full)
    os.replace(tmp, CKPT_PATH)


def load_checkpoint(default):
    if os.path.exists(CKPT_PATH):
        with np.load(CKPT_PATH) as data:
            arr = data['phi_corrected']
            if arr.shape == default.shape:
                return arr.copy()
    return default.copy()


def log_line(msg):
    print(msg, flush=True)
    with open(LOG_PATH, 'a') as f:
        f.write(msg + '\n')


init_csv_if_needed()
done = load_done_slices()
corrected_full = load_checkpoint(phi_full)
log_line(f'[start] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}  '
          f'already-done = {len(done)} / {D}')

In [ ]:
if SMOKE_TEST:
    if SMOKE_SLICE_INDICES is not None:
        target_slices = list(SMOKE_SLICE_INDICES)
    else:
        # Pick a quick informative trio: lightest, median, and heaviest folded slice.
        folded = np.where(fold_per_slice > 0)[0]
        if len(folded) == 0:
            target_slices = [0]
        else:
            order = np.argsort(fold_per_slice[folded])
            picks = [folded[order[0]], folded[order[len(order)//2]], folded[order[-1]]]
            target_slices = [int(z) for z in picks]
    log_line(f'[smoke] running slices {target_slices}')
else:
    target_slices = list(range(D))

for i, z in enumerate(target_slices):
    if z in done:
        continue
    t_slice = time.time()
    try:
        row = run_one_slice(corrected_full, z)
    except Exception as exc:
        tb = traceback.format_exc(limit=4).replace('\n', ' | ')
        log_line(f'[ERR  ] z={z}  {type(exc).__name__}: {exc} :: {tb}')
        # Record a placeholder row so we don't retry forever in a loop
        n_neg, mt = slice_fold_stats(corrected_full, z)
        row = {
            'z': int(z), 'H': int(H), 'W': int(W),
            'init_n_neg_tri': n_neg, 'init_min_tri': mt,
            'L2_n_neg_tri': -1, 'L2_min_tri': float('nan'),
            'L2_L1': float('nan'), 'L2_L2': float('nan'),
            'L2_t': 0.0, 'L2_exception': f'{type(exc).__name__}: {exc}',
            'L1_L1': float('nan'), 'L1_L2': float('nan'),
            'L1_n_neg_tri': -1, 'L1_min_tri': float('nan'),
            'L1_components': 0, 'L1_polished': 0,
            'L1_rejected': 0, 'L1_skipped_large': 0,
            'L1_iter_total': 0, 'L1_t': 0.0,
            'total_t': time.time() - t_slice, 'feasible': False,
            'l1_drop_pct': 0.0,
            'phi_l1': None,
        }
    # Update the corrected volume with the L1 result (if available).
    if row.get('phi_l1') is not None:
        corrected_full[1, z] = row['phi_l1'][0]
        corrected_full[2, z] = row['phi_l1'][1]
    row_to_write = {k: v for k, v in row.items() if k != 'phi_l1'}
    append_csv_row(row_to_write)
    done.add(z)
    log_line(f'[slice] z={z:>4d}  init={row["init_n_neg_tri"]:>5d}  '
              f'L2_neg={row["L2_n_neg_tri"]:>4d}  '
              f'L1_neg={row["L1_n_neg_tri"]:>4d}  '
              f'L2_L1={row["L2_L1"]:>9.3f}  '
              f'L1_L1={row["L1_L1"]:>9.3f}  '
              f'drop={row["l1_drop_pct"]:>5.1f}%  '
              f'L2_t={row["L2_t"]:>6.1f}s  L1_t={row["L1_t"]:>5.1f}s  '
              f'feasible={row["feasible"]}')
    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(corrected_full)
        log_line(f'[ckpt ] saved at i={i+1}, done={len(done)}/{D}')

save_checkpoint(corrected_full)
log_line(f'[end  ] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}  done={len(done)}/{D}')

## Aggregate stats

In [ ]:
df = pd.read_csv(CSV_PATH)
df_folded = df[df['init_n_neg_tri'] > 0].copy()
df_feasible = df_folded[df_folded['feasible']].copy()

print(f'total slices                            : {len(df)}')
print(f'slices with folds (work needed)         : {len(df_folded)}')
print(f'slices reaching feasibility             : {len(df_feasible)} / {len(df_folded)}'
      + (f'  ({100.0*len(df_feasible)/len(df_folded):.1f}%)' if len(df_folded) else ''))
print()
if len(df_feasible) > 0:
    print(f'L1 drop after polish (feasible slices only):')
    print(f'  mean = {df_feasible["l1_drop_pct"].mean():.2f}%   '
          f'median = {df_feasible["l1_drop_pct"].median():.2f}%   '
          f'max = {df_feasible["l1_drop_pct"].max():.2f}%')
    print(f'L2 phase wall time (feasible slices):')
    print(f'  total = {df_feasible["L2_t"].sum():.1f}s   '
          f'mean = {df_feasible["L2_t"].mean():.1f}s   '
          f'p95 = {df_feasible["L2_t"].quantile(0.95):.1f}s   '
          f'max = {df_feasible["L2_t"].max():.1f}s')
    print(f'L1 polish wall time (feasible slices):')
    print(f'  total = {df_feasible["L1_t"].sum():.1f}s   '
          f'mean = {df_feasible["L1_t"].mean():.1f}s   '
          f'p95 = {df_feasible["L1_t"].quantile(0.95):.1f}s   '
          f'max = {df_feasible["L1_t"].max():.1f}s')
    print(f'Wall time grand total: {df["total_t"].sum():.1f}s'
          f' = {df["total_t"].sum() / 3600:.2f} h')

In [ ]:
FIG_DIR = OUTPUT_DIR

# (1) per-slice fold overview
fig, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
ax.plot(df['z'], df['init_n_neg_tri'], lw=0.7, color='#c62828', label='initial n_neg_tri')
ax.plot(df['z'], df['L1_n_neg_tri'].clip(lower=0), lw=0.7, color='#1b8a3a',
         label='after L2+L1 (n_neg_tri)')
ax.set_yscale('symlog', linthresh=1)
ax.set_xlabel('z (slice index)'); ax.set_ylabel('n_neg_tri (symlog)')
ax.set_title('Folded triangles per slice: before vs after L2+L1')
ax.legend()
fig.savefig(os.path.join(FIG_DIR, 'per_slice_fold_overview.png'), dpi=200, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'per_slice_fold_overview.pdf'), bbox_inches='tight')
plt.show()

# (2) L1 vs L2 sparsity scatter
if len(df_feasible) > 0:
    fig, ax = plt.subplots(figsize=(6.5, 6), constrained_layout=True)
    lo = min(df_feasible['L1_L1'].min(), df_feasible['L2_L1'].min()) * 0.95
    hi = max(df_feasible['L1_L1'].max(), df_feasible['L2_L1'].max()) * 1.05
    ax.plot([lo, hi], [lo, hi], color='#888888', lw=1, linestyle='--')
    ax.scatter(df_feasible['L2_L1'], df_feasible['L1_L1'],
                s=8, alpha=0.7, color='#1b8a3a')
    ax.set_xlabel('L1-norm of correction after L2 phase'); ax.set_ylabel('L1-norm after L2+L1 phase')
    ax.set_title('L1 norm: L2 result vs L1-polished result (one point per feasible slice)\n'
                  'Points below the diagonal = L1 polish strictly improves the correction')
    ax.set_aspect('equal')
    fig.savefig(os.path.join(FIG_DIR, 'l1_vs_l2_sparsity.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'l1_vs_l2_sparsity.pdf'), bbox_inches='tight')
    plt.show()

# (3) time distribution
if len(df_folded) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), constrained_layout=True)
    axes[0].hist(df_folded['L2_t'], bins=60, color='#5b7fb5')
    axes[0].set_xlabel('L2 phase time (s)'); axes[0].set_ylabel('# slices')
    axes[0].set_title('L2 phase runtime per folded slice')
    axes[1].hist(df_folded['L1_t'], bins=60, color='#1b8a3a')
    axes[1].set_xlabel('L1 polish time (s)'); axes[1].set_ylabel('# slices')
    axes[1].set_title('L1 polish runtime per folded slice')
    fig.savefig(os.path.join(FIG_DIR, 'time_distribution.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'time_distribution.pdf'), bbox_inches='tight')
    plt.show()

# (4) worst-slice visual
if len(df_folded) > 0:
    worst_z = int(df_folded.sort_values('init_n_neg_tri', ascending=False).iloc[0]['z'])
    phi_init_slice = np.stack([phi_full[1, worst_z].copy(),
                                 phi_full[2, worst_z].copy()])
    phi_l1_slice = np.stack([corrected_full[1, worst_z].copy(),
                               corrected_full[2, worst_z].copy()])
    res = np.linalg.norm(phi_l1_slice - phi_init_slice, axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    T1i, T2i = _triangle_areas_2d(phi_init_slice[0], phi_init_slice[1])
    T1f, T2f = _triangle_areas_2d(phi_l1_slice[0], phi_l1_slice[1])
    init_min = np.minimum(T1i, T2i)
    final_min = np.minimum(T1f, T2f)
    vmax = max(abs(init_min.min()), abs(final_min.min()), 1.0)
    axes[0].imshow(init_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0].set_title(f'worst slice z={worst_z}: min(T1,T2) before')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    axes[1].imshow(final_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1].set_title('after L2+L1 polish')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    im2 = axes[2].imshow(res, cmap='Reds')
    axes[2].set_title('per-pixel correction magnitude')
    axes[2].set_xticks([]); axes[2].set_yticks([])
    fig.colorbar(im2, ax=axes[2], shrink=0.85)
    fig.savefig(os.path.join(FIG_DIR, 'worst_slice_visual.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'worst_slice_visual.pdf'), bbox_inches='tight')
    plt.show()

## Save final corrected volume

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'deformation3d_corrected_2d.npy')
np.save(final_path, corrected_full)
print(f'Saved corrected (per-slice 2D) volume to {final_path}')

summary = {
    'data_path': DATA_PATH,
    'shape': list(phi_full.shape),
    'threshold': THRESHOLD, 'eps_l1': EPS_L1,
    'L2_max_outer': L2_MAX_OUTER,
    'L2_max_per_index': L2_MAX_PER_INDEX,
    'L2_max_minimize': L2_MAX_MINIMIZE,
    'L1_polish_max_iter': L1_POLISH_MAX_ITER,
    'L1_touch_tol': L1_TOUCH_TOL,
    'L1_border': L1_BORDER,
    'L1_component_max_size': L1_COMPONENT_MAX_SIZE,
    'total_slices': int(len(df)),
    'slices_with_folds': int(len(df_folded)),
    'slices_feasible': int(len(df_feasible)),
    'l1_drop_pct_mean': float(df_feasible['l1_drop_pct'].mean()) if len(df_feasible) else 0.0,
    'l1_drop_pct_median': float(df_feasible['l1_drop_pct'].median()) if len(df_feasible) else 0.0,
    'total_wall_time_s': float(df['total_t'].sum()),
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved summary.json')